# <center> Get Genesets

## <center> Description

- env: squidpy_v06
- Get gene sets that will be used for enrichment tests.

## <center> Set-up

## Python

In [1]:
# Load libraries
from pathlib import Path
import numpy as np
import pandas as pd
import decoupler as dc

import sys
from pathlib import Path

# When this notebook is exported and run as a script, add the project root to sys.path
# so local workflow.* imports behave like they do in the notebook kernel.
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[3]
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

## helper functions
from workflow.scripts.funs.helper_funs import get_project_root

#### **Paths**

In [2]:
if 'snakemake' in locals():
    print('Hello World')
    # output
    # collectri=snakemake.output['collectri']
    # progeny=snakemake.output['progeny']
    # hallmark=snakemake.output['hallmark']
    # reactome=snakemake.output['reactome']
    # KEGG=snakemake.output['KEGG']
else:
    ## get paths
    ROOT = get_project_root(globals())
    ## output
    ### objects
    path_output_objects = ROOT / "output/preprocessing/objects/getGenesets"
    path_output_objects.mkdir(parents=True, exist_ok=True)
    out_collectri=path_output_objects / 'collectri.tsv'
    out_progeny=path_output_objects / 'progeny.tsv'
    out_go_cc=path_output_objects / 'go_cc.tsv'
    out_go_mf=path_output_objects / 'go_mf.tsv'
    out_go_bp=path_output_objects / 'go_bp.tsv'
    out_hallmark=path_output_objects / 'hallmark.tsv'
    out_reactome=path_output_objects / 'reactome.tsv'
    out_KEGG=path_output_objects / 'kegg.tsv'

## <center> Pathway enrichment and TF act

## Get resources

#### **CollecTRI TFs**

In [3]:
# get table

df_collectri = dc.op.collectri(organism='human', remove_complexes=False, license='academic')
## select cols
df_collectri=df_collectri[['source', 'target', 'weight']]

df_collectri.head()

,source,target,weight
0,MYC,TERT,1.0
1,SPI1,BGLAP,1.0
2,SMAD3,JUN,1.0
3,SMAD4,JUN,1.0
4,STAT5A,IL2,1.0


In [4]:
# save table
df_collectri.to_csv(out_collectri, sep='\t', index=False)

#### **Progeny pathways**

In [5]:
# get table 
df_progeny = dc.op.progeny(organism='human', top=500, thr_padj=0.05)

df_progeny.head()

,source,target,weight,padj
0,Androgen,TMPRSS2,11.490631,2.384806e-47
1,Androgen,NKX3-1,10.622551,2.205102e-44
2,Androgen,MBOAT2,10.472733,4.632376e-44
3,Androgen,KLK2,10.176186,1.944410e-40
4,Androgen,SARG,11.386852,2.790210e-40


In [6]:
## number of targets per pathway
df_progeny.groupby('source')['target'].count()

source
Androgen    499
EGFR        500
Estrogen    498
Hypoxia     499
JAK-STAT    499
MAPK        498
NFkB        500
PI3K        498
TGFb        500
TNFa        500
Trail       202
VEGF        499
WNT         273
p53         498
Name: target, dtype: int64

In [7]:
# save table
df_progeny.to_csv(out_progeny, sep='\t', index=False)

#### **GO, Hallmark, Reactome, KEGG from MSigDb**

In [8]:
# get all gene sets in MSigDB
gene_sets = dc.op.resource('MSigDB')
gene_sets.head()

,genesymbol,collection,geneset
,,,
0,A1BG,reactome_pathways,REACTOME_HEMOSTASIS
1,A1BG,go_cellular_component,GOCC_PLATELET_ALPHA_GRANULE_LUMEN
2,A1BG,chemical_and_genetic_perturbations,CHENG_IMPRINTED_BY_ESTRADIOL
3,A1BG,immunesigdb,GSE13522_WT_VS_IFNG_KO_SKING_T_CRUZI_Y_STRAIN_...
4,A1BG,immunesigdb,GSE25088_CTRL_VS_IL4_AND_ROSIGLITAZONE_STIM_MA...


In [9]:
## available gene sets
gene_sets['collection'].unique()

array(['reactome_pathways', 'go_cellular_component',
       'chemical_and_genetic_perturbations', 'immunesigdb',
       'tf_targets_legacy', 'cell_type_signatures',
       'go_biological_process', 'positional', 'mirna_targets_mirdb',
       'tf_targets_gtrf', 'cancer_modules', 'vaccine_response',
       'go_molecular_function', 'oncogenic_signatures',
       'cancer_cell_atlas', 'hallmark', 'pid_pathways', 'kegg_pathways',
       'human_phenotype_ontology', 'wikipathways',
       'cancer_gene_neighborhoods', 'mirna_targets_legacy',
       'biocarta_pathways', 'kegg_medicus_pathways'], dtype=object)

In [10]:
# get GO

## cellular component ------------
go_cc=gene_sets[gene_sets['collection']=='go_cellular_component'][['geneset', 'genesymbol']]
### Remove duplicated entries
go_cc=go_cc[~go_cc.duplicated(['geneset', 'genesymbol'])]

## molecular function ------------
go_mf=gene_sets[gene_sets['collection']=='go_molecular_function'][['geneset', 'genesymbol']]
### Remove duplicated entries
go_mf=go_mf[~go_mf.duplicated(['geneset', 'genesymbol'])]

## biological process ------------
go_bp=gene_sets[gene_sets['collection']=='go_biological_process'][['geneset', 'genesymbol']]
### Remove duplicated entries
go_bp=go_bp[~go_bp.duplicated(['geneset', 'genesymbol'])]

# save
go_cc.to_csv(out_go_cc, sep='\t', index=False)
go_mf.to_csv(out_go_mf, sep='\t', index=False)
go_bp.to_csv(out_go_bp, sep='\t', index=False)

In [11]:
# get Hallmark

df_hallmark = gene_sets[gene_sets.collection=='hallmark'][['geneset', 'genesymbol']]
## Remove duplicated entries
df_hallmark = df_hallmark[~df_hallmark.duplicated(['geneset', 'genesymbol'])]

# save
df_hallmark.to_csv(out_hallmark, sep='\t', index=False)

print(f"Number of sets: {len(np.unique(df_hallmark.geneset))}")

Number of sets: 50


In [12]:
# get Reactome

df_reactome = gene_sets[gene_sets.collection=='reactome_pathways'][['geneset', 'genesymbol']]
## remove duplicates
df_reactome = df_reactome[~df_reactome.duplicated(['geneset', 'genesymbol'])]

# save
df_reactome.to_csv(out_reactome, sep='\t', index=False)

print(f"Number of sets: {len(np.unique(df_reactome.geneset))}")

Number of sets: 1787


In [13]:
# get KEGG

df_KEGG = gene_sets[gene_sets.collection=='kegg_pathways'][['geneset', 'genesymbol']]
## remove duplicates
df_KEGG = df_KEGG[~df_KEGG.duplicated(['geneset', 'genesymbol'])]

# save
df_KEGG.to_csv(out_KEGG, sep='\t', index=False)

print(f"Number of sets: {len(np.unique(df_KEGG.geneset))}")

Number of sets: 186
